# Database migration notebook
Notebook för ETL-migrering med städning av databas.
Uppdatera ny version av SQLite databas fil och ersätta nuvarande SQLite databasfil med nyare version
Uppdatera music_warehouse.duckdb med nya rena databasen
----
Kolla först vilka tables i nya databasen som finns
Gör de ändringarna som behövs
Uppdatera men behåll gamla filen som backup. Namnger databasfilerna med  

`powerbi_warehouse_OLD.db` 
`music_warehouse_OLD.duckdb`

In [1]:
import duckdb
import os

In [3]:
# 1. Starta en tillfällig DuckDB i minnet
con = duckdb.connect()
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

# 2. Koppla upp mot din SQLite-fil
sqlite_path = "../data/powerbi_warehouse_cleaned.db"
con.execute(f"ATTACH '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")

# 3. Använd DuckDB:s standardiserade 'information_schema' för att hitta tabellerna
query = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_catalog = 'sqlite_db';
"""
tables_df = con.execute(query).df()

print("Här är de exakta tabellnamnen inuti din SQLite-fil:\n")
print(tables_df)

con.close()

Här är de exakta tabellnamnen inuti din SQLite-fil:

                  table_name
0              spotify_daily
1          historical_charts
2                    top_200
3              dim_geography
4            top_200_cleaned
5      dim_geography_cleaned
6  historical_charts_cleaned


In [6]:

# Inställningar
SQLITE_PATH = "../data/powerbi_warehouse_cleaned.db"
DUCKDB_PATH = "../data/music_warehouse.duckdb"

def rebuild_duckdb_from_sqlite():
    print(f"Startar total återuppbyggnad av {DUCKDB_PATH}...")
    
    # 1. Ta bort den gamla DuckDB-filen om den råkar finnas kvar
    if os.path.exists(DUCKDB_PATH):
        os.remove(DUCKDB_PATH)
        print("Rensade gammal DuckDB-data.")

    # 2. Skapa en helt ny anslutning
    con = duckdb.connect(DUCKDB_PATH)
    
    try:
        # 3. Ladda SQLite engine
        con.execute("INSTALL sqlite;")
        con.execute("LOAD sqlite;")
        
        # 4. Koppla till din SQLite Source of Truth
        con.execute(f"ATTACH '{SQLITE_PATH}' AS sqlite_db (TYPE SQLITE);")
        
        print("Migrerar och mappar tabeller till Silver-lagret...")
        
        # Här mappar jag till de rena namnen i SQLite till "silver" namn i DuckDB
        tables_to_migrate = {
            "spotify_daily": "silver_spotify_daily",
            "historical_charts_cleaned": "silver_historical_charts",
            "top_200_cleaned": "silver_top_200_historical",
            "dim_geography_cleaned": "dim_geography"
        }

        for source, target in tables_to_migrate.items():
            print(f" -> {source} -> {target}")
            con.execute(f"CREATE TABLE {target} AS SELECT * FROM sqlite_db.{source};")

        # 5. Återskapa guld-vy (så att dashboarden fungerar direkt)
        print("Återskapar gold_spotify_daily view...")
        con.execute("""
            CREATE VIEW gold_spotify_daily AS 
            SELECT s.*, g.country_name 
            FROM silver_spotify_daily s
            LEFT JOIN dim_geography g ON s.country = g.iso_code;
        """)

        print("\nKLART! Source of Truth är nu återställd i DuckDB.")
        
    except Exception as e:
        print(f"KRASCH under återuppbyggnad: {e}")
    finally:
        con.close()

if __name__ == "__main__":
    rebuild_duckdb_from_sqlite()

Startar total återuppbyggnad av ../data/music_warehouse.duckdb...
Rensade gammal DuckDB-data.
Migrerar och mappar tabeller till Silver-lagret...
 -> spotify_daily -> silver_spotify_daily
 -> historical_charts_cleaned -> silver_historical_charts
 -> top_200_cleaned -> silver_top_200_historical
 -> dim_geography_cleaned -> dim_geography
Återskapar gold_spotify_daily view...

KLART! Source of Truth är nu återställd i DuckDB.


In [7]:
def rescue_sales_data():
    print("Räddar sales-datan och lägger in den i vår rena databas...")
    
    # 1. Anslut till din befintliga, rena DuckDB
    con = duckdb.connect("../data/music_warehouse.duckdb")
    
    try:
        # Peka denna på din ursprungliga fil
        original_file_path = "../data/processed/historical_media_sales.parquet" 
        
        print(f"Laddar in data från: {original_file_path}")
        
        # 3. Skapa tabellen och läs in datan direkt från filen
        con.execute(f"""
            CREATE OR REPLACE TABLE silver_music_format_sales AS 
            SELECT * FROM '{original_file_path}';
        """)
        
        # 4. Validering
        count = con.execute("SELECT COUNT(*) FROM silver_music_format_sales").fetchone()[0]
        print(f"KLART! 'silver_music_format_sales' är nu inne med {count} rader.")
        
    except Exception as e:
        print(f"Något gick fel: {e}")
    finally:
        con.close()

if __name__ == "__main__":
    rescue_sales_data()

Räddar sales-datan och lägger in den i vår rena databas...
Laddar in data från: ../data/processed/historical_media_sales.parquet
KLART! 'silver_music_format_sales' är nu inne med 1351 rader.
